In [ ]:

BRANCH="SciDFM"

!rm -rf FoSpy

!git clone --single-branch --branch $BRANCH https://github.com/errthumt/FoSpy.git FoSpy

%cd FoSpy/SciDFM/sandbox

%pip install -U -r requirements.txt


Cloning into 'FoSpy'...
remote: Enumerating objects: 4974, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 4974 (delta 188), reused 171 (delta 150), pack-reused 4733 (from 4)
Receiving objects: 100% (4974/4974), 82.03 MiB | 12.09 MiB/s, done.
Resolving deltas: 100% (2738/2738), done.
/content/FoSpy/SciDFM/sandbox/FoSpy/SciDFM/sandbox/FoSpy/SciDFM/sandbox
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [8]:
import json
import re
from pathlib import Path
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    LlamaTokenizer,
)

INPUT_SOURCES = {
    "Ba2Zn5As6 Supplemental Information": (
        "Ba2Zn5As6_SI.txt",
        "Solid-State Material Science",
    )
}

MAX_TOKENS = 2048
TEMPERATURE = 0.1

MODEL_ID = "OpenDFM/SciDFM-MoE-A5.6B-v1.0"

INPUT_DIR = Path("inputs")
OUTPUT_FILE = Path("output/tokens.json")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = LlamaTokenizer.from_pretrained(
    MODEL_ID, use_fast=False, trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

GEN_CONFIG = GenerationConfig(
    do_sample=True,
    top_k=20,
    top_p=0.9,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_TOKENS,
    eos_token_id=tokenizer.eos_token_id,
)


def get_prompt(input_text, context="None Specified"):
    pr = f"""
    Extract the fundamental scientific information pertaining to the synthetic
    method and product characterization described in the following input. Return
    the result as a JSON-formattedlist of atomic, order-independent scientific information
    tokens. Each token must represent one fact, relationship or entity. Whenever
    possible, token values should be coersible to primitive data types,
    including separating quantities into a `value` and `units` fields. If
    specified, context refers to the scientific field or chemical category
    pertaining to the intended synthetic products.

    Rules:
    - Remove redundancy.
    - Normalize chemical formulas.
    - Normalize references to individual elements (outside chemical formulas) to their full element name.
    - Each token must be independent of sentence order.
    - Token meaning must be independent of narrative context.
    - A preamble before JSON output is optional, but must not contain any JSON-signaling characters such as {{ or [.
    - No further response characters are allowed after the JSON output.


    Input:
    \"\"\"
    {input_text}
    \"\"\"

    Context:
    \"\"\"
    {context}
    \"\"\"
    """
    return pr


out = {"tokens": {}, "full_responses": {}}
out_tokens = out["tokens"]
out_full = out["full_responses"]

for name, (file, context) in INPUT_SOURCES.items():
    # FIXED: Replaced 'with' context manager with proper .read_text() execution
    target_file = INPUT_DIR / file
    if not target_file.exists():
        print(f"ERROR: Input file missing at {target_file}")
        continue

    input_text = target_file.read_text()

    prompt = get_prompt(input_text, context)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, generation_config=GEN_CONFIG)

    # FIXED: Correctly decoded the full generation batch and sliced off the original prompt text
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt) :]

    out_full[name] = raw

    # FIXED: Allowed the regex pattern to capture either JSON objects {} or arrays [] securely
    matched = re.search(r"[\{\[].*[\}\]]", raw, flags=re.DOTALL)
    if not matched:
        print(f"ERROR: No JSON found in output for {name}")
        continue

    json_str = matched.group(0)

    try:
        out_tokens[name] = json.loads(json_str)
    except json.JSONDecodeError:
        print(f"ERROR: Found JSON-like block but failed to parse: {json_str}")

with OUTPUT_FILE.open("w") as f:
    json.dump(out, f, indent=4)

print(f"Processing complete! File saved cleanly to {OUTPUT_FILE}")


ImportError: cannot import name 'is_torch_fx_available' from 'transformers.utils.import_utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py)